# Chapter 11: Fine-tuning BERT - Medium Tasks

This notebook covers intermediate fine-tuning techniques: freezing layers, few-shot learning, and partial layer freezing.

## Setup

Run all cells in this section to set up the environment and load the data.

Before running these cells, review the concepts from the main Chapter 11 notebook.

### [Optional] - Installing Packages on Google Colab

If you are viewing this notebook on Google Colab, uncomment and run the following code to install dependencies.

**Note**: Use a GPU for this notebook. In Google Colab, go to Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4.

In [ ]:
# %%capture
# !pip install "datasets>=2.18.0,<3" "transformers>=4.41.0" "setfit>=1.1.0" "accelerate>=0.27.2"
# Note: After running this cell, restart the kernel before proceeding

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

### Import Libraries

In [23]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

### Load Data

In [3]:
# Load Rotten Tomatoes dataset
tomatoes = load_dataset("rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]

### Helper Functions

In [4]:
def compute_metrics(eval_pred):
    """Calculate F1 score"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    load_f1 = evaluate.load("f1")
    f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"f1": f1}

## Challenges

Complete the following tasks by implementing the starter code.

### Level: Medium

**About This Task:**

Freezing layers can speed up training and prevent overfitting. By freezing the BERT encoder and only training the classifier, we update fewer parameters.

#### Medium Task 1: Freeze All Layers Except Classifier Head

### Instructions

1. Load a fresh BERT model
2. Freeze all encoder and embedding layers
3. Keep only the classifier head trainable
4. Train the model with frozen layers
5. Compare training time and performance with full fine-tuning

Load model and tokenizer.

In [5]:
# Load Model and Tokenizer
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Freeze all layers except the classifier.

In [6]:
# Freeze all layers except classifier
for name, param in model.named_parameters():
    # Trainable classification head
    if name.startswith("classifier"):
        param.requires_grad = True
    # Freeze everything else
    else:
        param.requires_grad = False

Verify which parameters are frozen.

In [7]:
# Count trainable vs frozen parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {frozen_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

Total parameters: 108,311,810
Trainable parameters: 1,538
Frozen parameters: 108,310,272
Trainable %: 0.00%


List which parameters are trainable.

In [8]:
# Show trainable parameters
print("\nTrainable parameters:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: shape {param.shape}")


Trainable parameters:
  classifier.weight: shape torch.Size([2, 768])
  classifier.bias: shape torch.Size([2])


Tokenize the data.

In [9]:
# Tokenize data
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
    """Tokenize input data"""
    return tokenizer(examples["text"], truncation=True)

tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)

Map: 100%|██████████| 1066/1066 [00:00<00:00, 27653.33 examples/s]


Define training arguments.

In [10]:
# Training arguments
training_args = TrainingArguments(
    "model_frozen",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="epoch",
    report_to="none"
)

Create trainer and train.

In [11]:
# Trainer which executes the training process
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

 95%|█████████▍| 506/534 [00:14<00:00, 34.02it/s]

{'loss': 0.6863, 'grad_norm': 4.839053630828857, 'learning_rate': 1.2734082397003748e-06, 'epoch': 0.94}


100%|██████████| 534/534 [00:16<00:00, 32.05it/s]

{'train_runtime': 16.6607, 'train_samples_per_second': 511.985, 'train_steps_per_second': 32.052, 'train_loss': 0.6863976810755354, 'epoch': 1.0}


TrainOutput(global_step=534, training_loss=0.6863976810755354, metrics={'train_runtime': 16.6607, 'train_samples_per_second': 511.985, 'train_steps_per_second': 32.052, 'train_loss': 0.6863976810755354, 'epoch': 1.0})

Evaluate the frozen model.

In [12]:
# Evaluate
results = trainer.evaluate()
print(f"\nFrozen Model F1 Score: {results['eval_f1']:.4f}")

100%|██████████| 67/67 [00:03<00:00, 19.24it/s]


Frozen Model F1 Score: 0.6505


### Task 1a: Compare with Full Fine-tuning

Train a model with all layers unfrozen and compare.

In [13]:
# Load fresh model (all layers trainable)
model_full = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

# Count trainable parameters
total_full = sum(p.numel() for p in model_full.parameters())
trainable_full = sum(p.numel() for p in model_full.parameters() if p.requires_grad)

print(f"Full model trainable parameters: {trainable_full:,}")

/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Full model trainable parameters: 108,311,810


In [14]:
# Train full model
trainer_full = Trainer(
    model=model_full,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer_full.train()

 94%|█████████▍| 502/534 [00:56<00:03,  8.77it/s]

{'loss': 0.4093, 'grad_norm': 8.229025840759277, 'learning_rate': 1.2734082397003748e-06, 'epoch': 0.94}


100%|██████████| 534/534 [01:03<00:00,  8.43it/s]

{'train_runtime': 63.3343, 'train_samples_per_second': 134.682, 'train_steps_per_second': 8.431, 'train_loss': 0.4050178813577145, 'epoch': 1.0}


TrainOutput(global_step=534, training_loss=0.4050178813577145, metrics={'train_runtime': 63.3343, 'train_samples_per_second': 134.682, 'train_steps_per_second': 8.431, 'train_loss': 0.4050178813577145, 'epoch': 1.0})

In [15]:
# Evaluate full model
results_full = trainer_full.evaluate()
print(f"\nFull Model F1 Score: {results_full['eval_f1']:.4f}")

# Compare
print(f"\nComparison:")
print(f"Frozen classifier only: {results['eval_f1']:.4f}")
print(f"Full fine-tuning: {results_full['eval_f1']:.4f}")
print(f"Difference: {results_full['eval_f1'] - results['eval_f1']:.4f}")

100%|██████████| 67/67 [00:03<00:00, 18.49it/s]


Full Model F1 Score: 0.8547

Comparison:
Frozen classifier only: 0.6505
Full fine-tuning: 0.8547
Difference: 0.2042


### Questions

1. How much faster was training with frozen layers? (Look at training time)

2. What is the performance difference between frozen and full fine-tuning?

3. When would you choose to freeze layers instead of full fine-tuning?

**About This Task:**

SetFit enables few-shot learning by combining pre-trained sentence transformers with contrastive learning. It works well with very few examples.

#### Medium Task 2: Few-shot Learning with SetFit (16 examples)

### Instructions

1. Sample only 16 training examples (8 per class)
2. Load a SetFit model
3. Train with contrastive learning
4. Evaluate on full test set
5. Compare with supervised learning on 16 examples

Import SetFit libraries.

In [17]:
!pip install setfit

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 3.7 MB/s  0:00:03 eta 0:00:01
Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
  Attempting uninstall: transformers━━━━━━━━━━━━ 0/3 [tokenizers]
    Found existing installation: transformers 4.38.22m0/3 [tokenizers]
    Uninstalling transformers-4.38.2:90m━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [transformers]
      Successfully uninstalled transformers-4.38.2━━━━━━━━━━━━━━━━ 1/3 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [setfit]2m2/3 [setfit]rmers]


In [18]:
from setfit import sample_dataset, SetFitModel
from setfit import TrainingArguments as SetFitTrainingArguments
from setfit import Trainer as SetFitTrainer

Sample few-shot training data.

In [19]:
# Sample 16 examples per class (8 per class)
sampled_train_data = sample_dataset(tomatoes["train"], num_samples=8)

print(f"Sampled training size: {len(sampled_train_data)}")
print(f"Test size: {len(test_data)}")

Sampled training size: 16
Test size: 1066


View the sampled data.

In [20]:
# Check label distribution
labels = sampled_train_data["label"]
print(f"\nLabel distribution:")
print(f"Negative (0): {labels.count(0)}")
print(f"Positive (1): {labels.count(1)}")


Label distribution:
Negative (0): 8
Positive (1): 8


Load SetFit model.

In [26]:
# Load a pre-trained SentenceTransformer model
setfit_model = SetFitModel.from_pretrained("sentence-transformers/all-mpnet-base-v2")

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


Define training arguments for SetFit.

In [27]:
# Define training arguments
setfit_args = SetFitTrainingArguments(
    num_epochs=3,  # The number of epochs to use for contrastive learning
    num_iterations=20  # The number of text pairs to generate
)
setfit_args.eval_strategy = setfit_args.evaluation_strategy

Create SetFit trainer.

In [28]:
# Create trainer
setfit_trainer = SetFitTrainer(
    model=setfit_model,
    args=setfit_args,
    train_dataset=sampled_train_data,
    eval_dataset=test_data,
    metric="f1"
)

Map: 100%|██████████| 16/16 [00:00<00:00, 4328.49 examples/s]


Train the SetFit model.

In [29]:
# Training loop
setfit_trainer.train()

***** Running training *****
  Num unique pairs = 640
  Batch size = 16
  Num unique pairs = 640
  Batch size = 16
  Num epochs = 3
  Num epochs = 3
  0%|          | 0/120 [00:00<?, ?it/s]/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)
/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The

{'embedding_loss': 0.285, 'grad_norm': 2.082585573196411, 'learning_rate': 1.6666666666666667e-06, 'epoch': 0.03}


 42%|████▎     | 51/120 [00:08<00:10,  6.27it/s]

{'embedding_loss': 0.0592, 'grad_norm': 0.020303072407841682, 'learning_rate': 1.2962962962962964e-05, 'epoch': 1.25}


 84%|████████▍ | 101/120 [00:16<00:03,  6.31it/s]

{'embedding_loss': 0.0004, 'grad_norm': 0.006511565297842026, 'learning_rate': 3.7037037037037037e-06, 'epoch': 2.5}


100%|██████████| 120/120 [00:19<00:00,  6.21it/s]


{'train_runtime': 19.3318, 'train_samples_per_second': 99.318, 'train_steps_per_second': 6.207, 'train_loss': 0.02676927112042904, 'epoch': 3.0}


Evaluate SetFit model.

In [30]:
# Evaluate the model on our test data
setfit_results = setfit_trainer.evaluate()
print(f"\nSetFit F1 Score: {setfit_results['f1']:.4f}")

***** Running evaluation *****



SetFit F1 Score: 0.8380


Examine the classifier head.

In [31]:
# The classifier is a simple logistic regression
print("\nClassifier head:")
print(setfit_model.model_head)


Classifier head:
LogisticRegression()


### Task 2a: Compare with Standard Fine-tuning

Train BERT with the same 16 examples and compare.

In [32]:
# Train BERT with 16 examples
model_fewshot = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer_fewshot = AutoTokenizer.from_pretrained(model_id)

# Tokenize few-shot data
def preprocess_function(examples):
    return tokenizer_fewshot(examples["text"], truncation=True)

data_collator_fewshot = DataCollatorWithPadding(tokenizer=tokenizer_fewshot)
tokenized_fewshot_train = sampled_train_data.map(preprocess_function, batched=True)
tokenized_fewshot_test = test_data.map(preprocess_function, batched=True)

/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 1066/1066 [00:00<00:00, 23396.92 examples/s]


In [33]:
# Train with same arguments
trainer_fewshot = Trainer(
    model=model_fewshot,
    args=training_args,
    train_dataset=tokenized_fewshot_train,
    eval_dataset=tokenized_fewshot_test,
    tokenizer=tokenizer_fewshot,
    data_collator=data_collator_fewshot,
    compute_metrics=compute_metrics,
)
trainer_fewshot.train()

  0%|          | 0/1 [00:00<?, ?it/s]/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)
/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memor

{'train_runtime': 4.4993, 'train_samples_per_second': 3.556, 'train_steps_per_second': 0.222, 'train_loss': 0.853053867816925, 'epoch': 1.0}


TrainOutput(global_step=1, training_loss=0.853053867816925, metrics={'train_runtime': 4.4993, 'train_samples_per_second': 3.556, 'train_steps_per_second': 0.222, 'train_loss': 0.853053867816925, 'epoch': 1.0})

In [34]:
# Evaluate BERT few-shot
bert_fewshot_results = trainer_fewshot.evaluate()

print(f"\nComparison with 16 training examples:")
print(f"SetFit: {setfit_results['f1']:.4f}")
print(f"BERT:   {bert_fewshot_results['eval_f1']:.4f}")
print(f"Difference: {setfit_results['f1'] - bert_fewshot_results['eval_f1']:.4f}")

/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)
100%|██████████| 67/67 [00:03<00:00, 17.90it/s]


Comparison with 16 training examples:
SetFit: 0.8380
BERT:   0.0000
Difference: 0.8380


### Questions

1. How does SetFit perform compared to BERT with only 16 examples?

2. Why is SetFit effective for few-shot learning? (Hint: contrastive learning)

3. What is the minimum number of examples per class needed for SetFit to work?

**About This Task:**

You can freeze only the first N layers while keeping later layers trainable. This balances training speed and performance.

#### Medium Task 3: Partial Layer Freezing (Freeze First N Layers)

### Instructions

1. Freeze embeddings and first 5 encoder layers (layers 0-4)
2. Keep layers 5-11 and classifier trainable
3. Train and evaluate the model
4. Compare with fully frozen and fully trainable models
5. Experiment with different freeze points

Load a fresh model.

In [35]:
# Load model
model_partial = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer_partial = AutoTokenizer.from_pretrained(model_id)

/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Find the index where layer 5 starts.

In [36]:
# Find parameter indices
for index, (name, param) in enumerate(model_partial.named_parameters()):
    if "encoder.layer.5" in name:
        print(f"Layer 5 starts at index {index}: {name}")
        layer_5_start_index = index
        break

Layer 5 starts at index 85: bert.encoder.layer.5.attention.self.query.weight


Freeze everything before layer 5.

In [37]:
# Encoder block 5 starts at index we found
# Freeze everything before that block
for index, (name, param) in enumerate(model_partial.named_parameters()):
    if index < layer_5_start_index:
        param.requires_grad = False

Verify which layers are frozen.

In [38]:
# Count frozen/trainable parameters
total_partial = sum(p.numel() for p in model_partial.parameters())
trainable_partial = sum(p.numel() for p in model_partial.parameters() if p.requires_grad)
frozen_partial = total_partial - trainable_partial

print(f"\nPartial freezing stats:")
print(f"Total parameters: {total_partial:,}")
print(f"Trainable parameters: {trainable_partial:,}")
print(f"Frozen parameters: {frozen_partial:,}")
print(f"Trainable %: {100 * trainable_partial / total_partial:.2f}%")


Partial freezing stats:
Total parameters: 108,311,810
Trainable parameters: 50,207,234
Frozen parameters: 58,104,576
Trainable %: 46.35%


Show which encoder layers are frozen vs trainable.

In [39]:
# Show frozen/trainable layers
print("\nLayer status:")
for layer_num in range(12):
    for name, param in model_partial.named_parameters():
        if f"encoder.layer.{layer_num}." in name:
            status = "Trainable" if param.requires_grad else "Frozen"
            print(f"Layer {layer_num}: {status}")
            break


Layer status:
Layer 0: Frozen
Layer 1: Frozen
Layer 2: Frozen
Layer 3: Frozen
Layer 4: Frozen
Layer 5: Trainable
Layer 6: Trainable
Layer 7: Trainable
Layer 8: Trainable
Layer 9: Trainable
Layer 10: Trainable
Layer 11: Trainable


Tokenize data and train.

In [40]:
# Trainer which executes the training process
trainer_partial = Trainer(
    model=model_partial,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer_partial,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer_partial),
    compute_metrics=compute_metrics,
)
trainer_partial.train()

  0%|          | 0/534 [00:00<?, ?it/s]/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)
  0%|          | 2/534 [00:00<00:40, 13.14it/s]/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered 

{'loss': 0.4159, 'grad_norm': 7.822448253631592, 'learning_rate': 1.2734082397003748e-06, 'epoch': 0.94}


100%|█████████▉| 532/534 [00:38<00:00, 13.78it/s]Checkpoint destination directory model_frozen/checkpoint-534 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory model_frozen/checkpoint-534 already exists and is non-empty. Saving will proceed but saved results may be invalid.
100%|██████████| 534/534 [00:41<00:00, 13.01it/s]

{'train_runtime': 41.0419, 'train_samples_per_second': 207.836, 'train_steps_per_second': 13.011, 'train_loss': 0.4123854119232978, 'epoch': 1.0}


TrainOutput(global_step=534, training_loss=0.4123854119232978, metrics={'train_runtime': 41.0419, 'train_samples_per_second': 207.836, 'train_steps_per_second': 13.011, 'train_loss': 0.4123854119232978, 'epoch': 1.0})

Evaluate partial freezing.

In [41]:
# Evaluate
results_partial = trainer_partial.evaluate()
print(f"\nPartial Freeze F1 Score: {results_partial['eval_f1']:.4f}")

/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)
100%|██████████| 67/67 [00:04<00:00, 16.34it/s]


Partial Freeze F1 Score: 0.8424


### Task 3a: Compare All Freezing Strategies

Compare the three strategies: full training, full freezing, partial freezing.

In [42]:
# Summary comparison
print("\n=== Freezing Strategy Comparison ===")
print(f"\nFull Fine-tuning (all layers trainable):")
print(f"  F1 Score: {results_full['eval_f1']:.4f}")
print(f"  Trainable %: 100.00%")

print(f"\nPartial Freeze (layers 0-4 frozen):")
print(f"  F1 Score: {results_partial['eval_f1']:.4f}")
print(f"  Trainable %: {100 * trainable_partial / total_partial:.2f}%")

print(f"\nFull Freeze (only classifier trainable):")
print(f"  F1 Score: {results['eval_f1']:.4f}")
print(f"  Trainable %: {100 * trainable_params / total_params:.2f}%")


=== Freezing Strategy Comparison ===

Full Fine-tuning (all layers trainable):
  F1 Score: 0.8547
  Trainable %: 100.00%

Partial Freeze (layers 0-4 frozen):
  F1 Score: 0.8424
  Trainable %: 46.35%

Full Freeze (only classifier trainable):
  F1 Score: 0.6505
  Trainable %: 0.00%


### Questions

1. How does partial freezing performance compare to full fine-tuning and full freezing?

2. Why might freezing early layers work well? (Hint: early layers learn general features)

3. What freeze point (layer 0-11) would you choose for best speed/performance tradeoff?